# UK Wind Portfolio: Day-Ahead Generation, Price and Revenue

<a href="https://colab.research.google.com/github/Lisae04/uk-wind-revenue-forecast/blob/main/notebooks/uk_wind_revenue_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab walkthrough of the [`uk-wind-revenue-forecast`](https://github.com/Lisae04/uk-wind-revenue-forecast) pipeline for the 156 MW portfolio from the BREE 495 capstone (GA_07 V3, option 1).

| Step | What it does |
|---|---|
| 1 | Set up: mount Drive, get the code, install packages |
| 2 | Settings: GA option, data mode, test period, hub height |
| 3 | Sites: 26 cells → farms, Global Wind Atlas means, IEC class |
| 4 | Data: ERA5, archived NWP forecasts, Elexon prices and forecasts (cached on Drive) |
| 5 | Generation: physical power curve + LightGBM residual correction |
| 6 | Price: naive, LEAR, LightGBM + SHAP, Diebold-Mariano |
| 7 | Revenue: capture price, cannibalisation, forecast value |
| 8 | Save tables and figures to Drive |

**Run order:** Runtime → Run all. A real-data run downloads about 2.75 years of hourly data the first time (30–60 min). Later runs reuse the cache in `MyDrive/uk_wind_cache`.

## 1. Setup

In [ ]:
# @title Mount Drive, get the code, install packages
import os, sys, subprocess, zipfile, glob, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lisae04/uk-wind-revenue-forecast.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
    REPO = Path("/content/uk-wind-revenue-forecast")
else:                                     # local Jupyter: run from the repo's notebooks/ folder
    DRIVE = None
    REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

def _clone(url):
    return subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(REPO)],
                          capture_output=True, text=True).returncode == 0

if not (REPO / "src" / "ukwind").exists():
    ok = _clone(REPO_URL)                               # 1) public repo
    if not ok and IN_COLAB:                             # 2) private repo: Colab secret GITHUB_TOKEN
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            ok = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        except Exception as e:
            print("No usable GITHUB_TOKEN secret:", type(e).__name__)
    if not ok and DRIVE is not None:                    # 3) zip of the repo saved in Drive
        zips = sorted(DRIVE.glob("**/uk-wind-revenue-forecast*.zip"))
        if zips:
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall("/content")
            ok = (REPO / "src" / "ukwind").exists()
            print("Code from", zips[0])
    if not ok:
        raise RuntimeError("Could not get the code: make the repo public, add a GITHUB_TOKEN "
                           "Colab secret, or put uk-wind-revenue-forecast.zip in My Drive.")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO / "src"))
print("Code:", REPO)

## 2. Settings

- **`DATA_MODE`**: `real` downloads Open-Meteo and Elexon data (no API keys); `synthetic` is an offline demo whose numbers are not findings.
- **`GWA_CSV`**: the capstone C1 export. Left blank, the cell searches My Drive for `england_wind.csv`.
- **`HUB_HEIGHT_M`**: 100 m reproduces the GA assumption; above 100 m applies hourly shear extrapolation.

In [ ]:
# @title Settings
import pandas as pd
from ukwind.config import load_config

DATA_MODE   = "real"        # @param ["real", "synthetic"]
GA_OPTION   = 1             # @param [1, 2, 3, 4, 5]
START       = "2024-01-01"  # @param {type:"string"}
END         = "2026-09-30"  # @param {type:"string"}
TEST_START  = "2025-10-01"  # @param {type:"string"}
HUB_HEIGHT_M = 100          # @param {type:"slider", min:80, max:150, step:5}
GWA_CSV     = ""            # @param {type:"string"}

cfg = load_config(REPO / "config.yaml")
cfg["period"] = {"start": START, "end": END, "test_start": TEST_START}
cfg["portfolio"]["option"] = int(GA_OPTION)
cfg["turbine"]["hub_height_m"] = float(HUB_HEIGHT_M)
SYNTHETIC = DATA_MODE == "synthetic"

# cache and results on Drive, so downloads survive the Colab session
base = (DRIVE / "uk_wind_cache") if DRIVE is not None else (REPO / "data")
out = (DRIVE / "uk_wind_results") if DRIVE is not None else (REPO / "results")
cfg["paths"]["cache"] = base / "cache"
cfg["paths"]["processed"] = base / "processed"
cfg["paths"]["results"] = out / ("synthetic_demo" if SYNTHETIC else "real")
for p in cfg["paths"].values():
    Path(p).mkdir(parents=True, exist_ok=True)

if not SYNTHETIC:
    if GWA_CSV:
        gwa = Path(GWA_CSV)
    else:
        default = DRIVE / "BREE495_C1_Yield_England/outputs/england_wind.csv" if DRIVE else None
        hits = [default] if default and default.exists() else \
               (sorted(DRIVE.glob("**/england_wind.csv")) if DRIVE else [REPO / "data/raw/england_wind.csv"])
        if not hits:
            raise FileNotFoundError("england_wind.csv not found; set GWA_CSV")
        gwa = hits[0]
    if not gwa.exists():
        raise FileNotFoundError(gwa)
    cfg["portfolio"]["gwa_file"] = gwa
    print("GWA file:", gwa)

if not pd.Timestamp(START) < pd.Timestamp(TEST_START) < pd.Timestamp(END):
    raise ValueError("Need START < TEST_START < END")
import json
print(json.dumps({k: cfg[k] for k in ("period", "portfolio", "turbine")}, indent=1, default=str))

## 3. Sites

Each 4-character OS grid reference is a 1 km cell with one 6 MW turbine. Cell centres are converted British National Grid → WGS84, the GWA mean wind is attached, and the IEC class whose reference mean wind V_ave (7.5 / 8.5 / 10 m/s) is nearest is chosen.

In [ ]:
# @title Load data and build the site tables
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from ukwind import pipeline as pl
from ukwind.sites import summarise

pd.set_option("display.float_format", "{:,.3f}".format)
cells, farms, weather, system, refs, mode = pl.load_data(cfg, synthetic=SYNTHETIC)
print(summarise(cells, farms))
print("ML reference mode:", mode, "| references:", list(refs))
display(farms)

In [ ]:
# @title Map of the portfolio
try:
    import folium
    m = folium.Map(location=[cells.lat.mean(), cells.lon.mean()], zoom_start=7, tiles="cartodbpositron")
    for c in cells.itertuples():
        folium.CircleMarker([c.lat, c.lon], radius=4, weight=1, fill=True, fill_opacity=0.8,
                            tooltip=f"{c.cell_id} | farm {c.farm_id} | GWA {c.gwa_ms:.2f} m/s | IEC {c.iec_class}").add_to(m)
    for f in farms.itertuples():
        folium.Marker([f.lat, f.lon], tooltip=f"{f.name}: {f.mw:.0f} MW, GA CF {f.ga_cf:.3f}").add_to(m)
    display(m)
except ImportError:
    plt.scatter(cells.lon, cells.lat, c=cells.gwa_ms); plt.colorbar(label="GWA m/s"); plt.show()

## 4. Data quality

Missing values remaining after cleaning (physical bounds, then gaps of up to 3 h interpolated). The price is missing wherever the Market Index traded zero volume.

In [ ]:
# @title Missing-value report
qa = {f"weather {fid}": w.isna().mean().mul(100).round(2) for fid, w in weather.items()}
qa = pd.DataFrame(qa).T
sys_qa = system.isna().mean().mul(100).round(2).rename("system (%)")
display(Markdown("**Weather, % missing per column**")); display(qa)
display(Markdown("**System table, % missing**")); display(sys_qa.to_frame())
fig, ax = plt.subplots(figsize=(9, 2.8))
system["price"].plot(ax=ax, lw=0.4, color="#2b6cb0"); ax.set_ylabel("GBP/MWh"); ax.set_title("GB Market Index price (hourly)")
ax.axvline(pd.Timestamp(TEST_START, tz="UTC"), color="#d9822b", ls="--", label="test start"); ax.legend(); plt.show()

## 5. Generation

$$U_{hub}=U_{100}\left(\tfrac{z_{hub}}{100}\right)^{\alpha_t},\qquad U_c=k_c\,U_{hub},\qquad k_c=\frac{\overline{GWA}_c}{\overline{U}_{hub,ERA5}}$$

$$U_{eq}=U_c\left(\tfrac{\rho}{1.225}\right)^{1/3},\qquad E_c=6\,\text{MW}\cdot f_{IEC}(U_{eq})\cdot(1-0.23)$$

The LightGBM correction learns $r_t = CF^{obs}_t - CF^{phys}_t$ on a metered reference farm whose own physical model is level-matched on training data only. It is refitted every 30 days, and every prediction is out of sample.

In [ ]:
# @title Run the generation model
G = pl.run_generation(cfg, cells, farms, weather, refs)
display(Markdown("**Long-term check: ERA5-driven CF vs GA CF (same turbine, same losses)**")); display(G["check"])
display(Markdown("**Reference farm, out of sample (CF units)**")); display(G["ref_eval"])
display(Markdown("**Portfolio vs ERA5-driven proxy** (no metered output exists for planned farms)")); display(G["port_eval"])
display(Image(str(cfg["paths"]["results"] / "figures" / "generation_week.png")))

## 6. Price

Information set at the gate (D-1 09:00 London): Elexon demand and wind forecast vintages published before the gate, the price 48 h and 168 h earlier, the D-2 daily mean and the D-1 morning mean, plus calendar features. The next cell asserts that no price feature comes from after the gate.

In [ ]:
# @title Run the price models (LEAR takes a few minutes)
P = pl.run_price(cfg, system)
display(Markdown("**Out-of-sample price errors (GBP/MWh); rMAE relative to naive_D-7**")); display(P["table"])
display(Markdown("**Diebold-Mariano p-values (column more accurate than row)**")); display(P["dm"])
display(Markdown("**Mean |SHAP| of LightGBM features (GBP/MWh)**")); display(P["shap"].to_frame())
for f in ("price_week.png", "price_mae_by_hour.png", "price_shap.png"):
    display(Image(str(cfg["paths"]["results"] / "figures" / f)))

## 7. Revenue

| Metric | Formula |
|---|---|
| Revenue | Σ E_t · P_t |
| Capture price | Σ E_t P_t / Σ E_t |
| Capture rate | capture price / mean P_t (missing when baseload < £10/MWh) |
| Capture discount | mean P_t − capture price |
| Forecast value | MAE of daily expected vs realised revenue, for each generation × price model |

In [ ]:
# @title Revenue, cannibalisation and forecast value
R = pl.run_revenue(cfg, G, P)
r = R["realised"]
display(Markdown(
    f"- Energy **{r['energy_mwh']/1e3:,.1f} GWh**, revenue **£{r['revenue_gbp']/1e6:,.2f} m**\n"
    f"- Capture price **{r['capture_price']:.2f}** vs baseload **{r['baseload_price']:.2f} GBP/MWh** "
    f"(discount {r['capture_discount']:.2f})\n"
    f"- {r['energy_at_negative_price_mwh']:,.0f} MWh produced over {r['negative_price_hours']} negative-price hours\n"
    f"- Price change per +10 pp forecast wind share: **{R['cannibalisation']['slope_gbp_per_10pp']:.1f} GBP/MWh** "
    f"(R² {R['cannibalisation']['r2']:.2f})"))
display(R["monthly"].astype(float))
display(Markdown("**Daily revenue forecast error**")); display(R["forecast_value"])
for f in ("capture_monthly.png", "revenue_daily.png"):
    display(Image(str(cfg["paths"]["results"] / "figures" / f)))

## 8. Save

In [ ]:
# @title Write summary.md and all tables to Drive
path = pl.write_report(cfg, farms, cells, G, P, R, mode, SYNTHETIC)
print("Saved to", cfg["paths"]["results"])
for p in sorted(cfg["paths"]["results"].rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(cfg["paths"]["results"]))
display(Markdown(path.read_text()))

---
**Notes**
- With `DATA_MODE = "synthetic"`, outputs go to `uk_wind_results/synthetic_demo/` and the summary is labelled synthetic.
- To use metered reference farms, run `!python {REPO}/scripts/find_wind_bmus.py --check`, then add BM units under `cfg["reference_farms"]["bmus"]` in the Settings cell (`bmu`, `lat`, `lon`, `capacity_mw`).
- The limitations (simulated portfolio "actuals", the APX Market Index as price, merchant-only revenue) are in the repo README.